In [ ]:
# from google.colab import drive
# drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
import random

/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(no):
    torch.manual_seed(no)
    random.seed(no)
    np.random.seed(no)
    os.environ['PYTHONHASHSEED'] = str()
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(100)

In [3]:
print(torch.__version__)

1.13.1


In [ ]:
# cd "drive/MyDrive/DinoV2/"

/content/drive/MyDrive/DinoV2


In [ ]:
# ls

0.DinoV2_Demo.ipynb  cat_dog_dataset/  DinoV2_Classification.ipynb


In [13]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/home/osero/Downloads/cats_dogs/'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'test']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=6,
                                             shuffle=True, num_workers=4)
              for x in ['train', 'test']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [14]:
dataloaders

{'train': <torch.utils.data.dataloader.DataLoader at 0x7fccc5760370>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7fcccd3fd870>}

Model


In [7]:
# load dino model
dinov2_vits14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


In [8]:
class DinoVisionTransformerClassifier(nn.Module):
    def __init__(self):
        super(DinoVisionTransformerClassifier, self).__init__()
        self.transformer = dinov2_vits14
        self.classifier = nn.Sequential(
            nn.Linear(384, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
    
    def forward(self, x):
        x = self.transformer(x)
        x = self.transformer.norm(x)
        x = self.classifier(x)
        return x


In [9]:
import torch.optim as optim

model = DinoVisionTransformerClassifier()


model1 = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model1.fc.in_features
model1.fc = nn.Linear(num_ftrs, 2)
model1 = model1.to(device)


criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
optimizer = optim.Adam(model.parameters(), lr=0.000001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/osero/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:04<00:00, 10.7MB/s]


In [15]:
len(dataloaders["train"])

1335

In [16]:
model = model.to(device)

Train

In [17]:
for epoch in range(6):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(dataloaders["train"], 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 50:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,    50] loss: 0.036
[1,   100] loss: 0.033
[1,   150] loss: 0.058
[1,   200] loss: 0.068
[1,   250] loss: 0.046
[1,   300] loss: 0.047
[1,   350] loss: 0.033
[1,   400] loss: 0.025
[1,   450] loss: 0.032
[1,   500] loss: 0.053
[1,   550] loss: 0.036
[1,   600] loss: 0.024
[1,   650] loss: 0.035
[1,   700] loss: 0.086
[1,   750] loss: 0.040
[1,   800] loss: 0.065
[1,   850] loss: 0.026
[1,   900] loss: 0.024
[1,   950] loss: 0.026
[1,  1000] loss: 0.039
[1,  1050] loss: 0.028
[1,  1100] loss: 0.076
[1,  1150] loss: 0.027
[1,  1200] loss: 0.031
[1,  1250] loss: 0.035
[1,  1300] loss: 0.034
[2,    50] loss: 0.014
[2,   100] loss: 0.037
[2,   150] loss: 0.024
[2,   200] loss: 0.047
[2,   250] loss: 0.036
[2,   300] loss: 0.047
[2,   350] loss: 0.037
[2,   400] loss: 0.018
[2,   450] loss: 0.031
[2,   500] loss: 0.036
[2,   550] loss: 0.020
[2,   600] loss: 0.022
[2,   650] loss: 0.014
[2,   700] loss: 0.045
[2,   750] loss: 0.063
[2,   800] loss: 0.027
[2,   850] loss: 0.036
[2,   900] 

Testing

In [18]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in dataloaders["test"]:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = model(images.to(device))
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted.to("cpu") == labels).sum().item()

print(f'Accuracy of the network on the {len(dataloaders["test"])*6} test images: {100 * correct // total} %')

Accuracy of the network on the 2028 test images: 99 %


Gradcam not possible as conv layer is not extrating much features (or) shall we try on the first conv layer in arch

Resnet classification

In [19]:
model1 = model1.to(device)

for epoch in range(6):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(dataloaders["train"], 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model1(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 50:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,    50] loss: 0.769
[1,   100] loss: 0.797
[1,   150] loss: 0.793
[1,   200] loss: 0.771
[1,   250] loss: 0.808
[1,   300] loss: 0.767
[1,   350] loss: 0.790
[1,   400] loss: 0.792
[1,   450] loss: 0.773
[1,   500] loss: 0.795
[1,   550] loss: 0.799
[1,   600] loss: 0.806
[1,   650] loss: 0.772
[1,   700] loss: 0.785
[1,   750] loss: 0.796
[1,   800] loss: 0.818
[1,   850] loss: 0.771
[1,   900] loss: 0.785
[1,   950] loss: 0.778
[1,  1000] loss: 0.796
[1,  1050] loss: 0.798
[1,  1100] loss: 0.781
[1,  1150] loss: 0.765
[1,  1200] loss: 0.766
[1,  1250] loss: 0.763
[1,  1300] loss: 0.772
[2,    50] loss: 0.787
[2,   100] loss: 0.772
[2,   150] loss: 0.770
[2,   200] loss: 0.791
[2,   250] loss: 0.791
[2,   300] loss: 0.791
[2,   350] loss: 0.775
[2,   400] loss: 0.796
[2,   450] loss: 0.783
[2,   500] loss: 0.809
[2,   550] loss: 0.776
[2,   600] loss: 0.804
[2,   650] loss: 0.743
[2,   700] loss: 0.806
[2,   750] loss: 0.792
[2,   800] loss: 0.796
[2,   850] loss: 0.797
[2,   900] 

In [20]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in dataloaders["test"]:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = model1(images.to(device))
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted.to("cpu") == labels).sum().item()

print(f'Accuracy of the network on the {len(dataloaders["test"])*6} test images: {100 * correct // total} %')

Accuracy of the network on the 2028 test images: 38 %


In [21]:
correct

785

In [22]:
total

2023